In [4]:
# ============================================================
# INGEST DRIVERS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType, DateType

In [5]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_LANDING
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:07:46 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:07:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:07:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:07:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:07:47 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [ ]:
# -----------------------------------------
# 2. IMPORT HELPER FUNCTIONS
# -----------------------------------------
try:
    add_ingestion_metadata
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    helper_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb"
    with open(helper_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

✔ Bronze helper functions loaded


In [7]:
# -----------------------------------------
# 3. Define source + target paths
# -----------------------------------------
source_file = f"{BRONZE_LANDING}/drivers.json"
target_folder = f"{BRONZE_ROOT}/drivers"
target_file = f"{target_folder}/drivers.parquet"

os.makedirs(target_folder, exist_ok=True)

print("Source:", source_file)
print("Target:", target_file)

Source: /Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json
Target: /Users/manoharazalki/F1-ANALYTICS/data/bronze/drivers/drivers.parquet


In [8]:
# -----------------------------------------
# 4. Define schema (nested structure)
# -----------------------------------------
name_schema = StructType([
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True),
])

drivers_schema = StructType([
    StructField("driverId", StringType(), True),
    StructField("name", name_schema, True),
    StructField("dateOfBirth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [9]:
# -----------------------------------------
# 5. Read JSON
# -----------------------------------------
drivers_df = (
    spark.read
        .format("json")
        .schema(drivers_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)

print("✔ Drivers file read successfully")
drivers_df.show(5, truncate=False)

✔ Drivers file read successfully
+---------+-------------------+-----------+-----------+----------------------------------------------+
|driverId |name               |dateOfBirth|nationality|url                                           |
+---------+-------------------+-----------+-----------+----------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |
|acheson  |{kenny, acheson}   |1957-11-27 |british    |http://en.wikipedia.org/wiki/Kenny_Acheson    |
|adams    |{philippe, adams}  |1969-11-19 |belgian    |http://en.wikipedia.org/wiki/Philippe_Adams   |
|ader     |{walt, ader}       |1913-12-15 |american   |http://en.wikipedia.org/wiki/Walt_Ader        |
+---------+-------------------+-----------+-----------+----------------------------------------------+
only showing top 5 rows


In [10]:
# -----------------------------------------
# 6. Add metadata
# -----------------------------------------
drivers_final_df = add_ingestion_metadata(drivers_df, source_file)

print("✔ Metadata added")
drivers_final_df.show(5, truncate=False)

✔ Metadata added
+---------+-------------------+-----------+-----------+----------------------------------------------+------------------------+------------------------------------------------------------------+
|driverId |name               |dateOfBirth|nationality|url                                           |ingest_timestamp        |source_file                                                       |
+---------+-------------------+-----------+-----------+----------------------------------------------+------------------------+------------------------------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|2026-06-08 23:07:49.1794|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |2026-06-08 23:07:49.1794|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|acheson

In [11]:
# -----------------------------------------
# 7. Write to Bronze (Option C)
# -----------------------------------------
(
    drivers_final_df
        .write
        .mode("overwrite")
        .parquet(target_file)
)

print(f"✔ Drivers Bronze written to: {target_file}")

✔ Drivers Bronze written to: /Users/manoharazalki/F1-ANALYTICS/data/bronze/drivers/drivers.parquet


In [12]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(target_file).show(5, truncate=False)

+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+------------------------------------------------------------------+
|driverId |name               |dateOfBirth|nationality|url                                           |ingest_timestamp          |source_file                                                       |
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+------------------------------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|acheson  |{ken